# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all entities by their `@id`. Analyses include data overview, extraction, processing, and a simple visualization.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant's Dataset class
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets and fields. The Croissant schema defines record sets and their fields, each with a unique `@id`. We print their keys and IDs to understand the dataset organization.

In [ ]:
# List all available record sets and their @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in this Croissant dataset.")
else:
    print("Record Sets and their field @ids:")
    for rs in record_sets:
        print(f"- Record Set: {rs['@id']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            for field in fields:
                print(f"    - Field @id: {field['@id']} ({field.get('name', '')})")
        else:
            print("    - No fields found")

## 3. Data Extraction

Load the main clinical record set into a pandas DataFrame. Use the actual record set and field `@id`s gathered from the overview.

Note: For this dataset, we'll extract all main record sets. The example shown loads the first non-empty record set.


In [ ]:
# Identify the main record set(s) to extract
main_record_sets = [rs['@id'] for rs in dataset.record_sets if rs.get('field')]
print("Available record sets with fields:", main_record_sets)

# We'll extract data for the first main record set
if main_record_sets:
    main_rs_id = main_record_sets[0]
    print(f"Using main record set: {main_rs_id}")
    
    # Load data to DataFrame
    df = pd.DataFrame(list(dataset.records(record_set=main_rs_id)))
    print(f"Columns in record set {main_rs_id}:")
    print(df.columns.tolist())
    display(df.head())
else:
    df = pd.DataFrame()
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing: filter numeric fields, normalize, and group-by analyses. All fields will be referenced by their `@id`.

*If you want to analyze other fields, update the variables below to match the `@id`s and column names shown above.*

In [ ]:
if not df.empty:
    # Choose a numeric field and a group field for demonstration
    # Let's pick 'Age at diagnosis of second CRC' and 'Sex' if present
    possible_numeric_fields = [col for col in df.columns if ('age' in col.lower() or df[col].dtype in ['int64', 'float64'])]
    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]  # reference by @id (column name)
    else:
        numeric_field_id = df.columns[0]  # fallback

    print(f"Numeric field selected for analysis: {numeric_field_id}")

    # Filter records with value above a threshold
    threshold = 50
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} ({len(filtered_df)}/{len(df)} records):")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (e.g., Sex)
    possible_group_fields = [col for col in df.columns if (col.lower().startswith('sex') or 'gender' in col.lower())]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Here, we plot the distribution of the main numeric field and, if available, differences by a grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Group plot if group_field exists
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIR² dataset package using `mlcroissant`, referencing all data entities by their `@id`. We overviewed available record sets, loaded main clinical data, ran exploratory analysis, and visualized numeric distributions. For further analyses, consider reviewing variable definitions and study documentation provided in the Croissant schema.

#### Key takeaways:
- The Croissant format allows principled, interoperable access to biomedical datasets.
- All fields and record sets are referenced using their unique `@id` values for reproducibility.
- Pandas and visualization tools can be seamlessly used for deeper data analysis workflows.